# Music Annotation Project V8.8E
## Notebook 56: Supervised Registered-Replay Pilot Reviewer

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook consumes the exact successful Notebook 55 shadow run. It presents the controlled Notebook 09 **proposal** beside the V8.8 registered-replay corrected page and requires a human page-level decision.

Available decisions:

- **Accept V8.8 correction**
- **Retain Notebook 09 proposal**
- **Defer**
- **Require upstream mask review**

Final closure is blocked while any page is unreviewed or deferred. This notebook records source identities, disagreement counts, decision history, approved output hashes and a curated upload package. It has no automatic approval or production-promotion authority.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy gradio pillow


## 0. Configuration and helpers

For the definitive run, set `NOTEBOOK55_RUN_OVERRIDE` to the exact successful Notebook 55 directory. Leaving the override as `None` selects the latest complete passed run.


In [ ]:
import cv2
import gradio as gr
import hashlib
import json
import shutil
import zipfile

import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

N53_ROOT = ROOT / 'm5_remove_operational_builder_v8_8b'
N55_ROOT = ROOT / 'm5_remove_notebook09_shadow_v8_8d'
OUTPUT_ROOT = ROOT / 'm5_remove_supervised_pilot_v8_8e'

NOTEBOOK53_RUN_OVERRIDE = None
NOTEBOOK55_RUN_OVERRIDE = None
PUBLIC_LINK = True

EXPECTED_N55_VERDICT = (
    'PASS_REGISTERED_REPLAY_NOTEBOOK09_'
    'SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT'
)

SUPPORTED = {
    '.png', '.jpg', '.jpeg',
    '.tif', '.tiff', '.bmp'
}


def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def read_json(path):
    path = Path(path)
    require(path.exists(), f'Missing JSON file: {path}')
    try:
        value = json.loads(path.read_text())
    except Exception as error:
        raise RuntimeError(
            f'Could not read JSON file {path}: {error}'
        ) from error
    require(isinstance(value, dict), f'JSON root must be an object: {path}')
    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, sort_keys=True, default=str)
    )
    temporary.replace(path)


def sha256_file(path):
    path = Path(path)
    require(path.exists() and path.is_file(), f'Missing file for SHA-256: {path}')
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def array_hash(array):
    array = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode())
    digest.update(str(array.shape).encode())
    digest.update(array.tobytes())
    return digest.hexdigest()


def latest_complete(root, required):
    root = Path(root)
    hits = []
    if root.exists():
        for path in root.rglob('*'):
            if path.is_dir() and all((path / item).exists() for item in required):
                hits.append(path)
    return max(hits, key=lambda path: path.stat().st_mtime) if hits else None


def imread(path):
    path = Path(path)
    require(path.exists() and path.is_file(), f'Missing image: {path}')
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Unreadable image: {path}')
    return image


RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook56_v8_8e_pilot'
OUT = OUTPUT_ROOT / RUN
DEC = OUT / 'decisions'
FINAL = OUT / 'final'
T = OUT / 'tables'
A = OUT / 'audit'
U = OUT / 'upload'

for path in [OUT, DEC, FINAL, T, A, U]:
    path.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)


## 1. Resolve the passed Notebook 55 run and its exact sources


In [ ]:
N55 = (
    Path(NOTEBOOK55_RUN_OVERRIDE)
    if NOTEBOOK55_RUN_OVERRIDE
    else latest_complete(
        N55_ROOT,
        [
            'notebook55_verdict.json',
            'tables/shadow_page_comparison.csv',
            'tables/roster_audit.json',
        ],
    )
)

require(N55 is not None, 'Complete Notebook 55 run not found.')

v55 = read_json(N55 / 'notebook55_verdict.json')

require(
    v55.get('verdict') == EXPECTED_N55_VERDICT,
    (
        'Notebook 55 did not pass the required registered-replay '
        f'shadow gate: {v55}'
    ),
)

require(
    v55.get('first_failed_gate') is None,
    'Notebook 55 records a failed gate.',
)

required_v55_keys = {
    'notebook53_run',
    'notebook09_run',
    'policy_id',
    'implementation_id',
}
missing_v55_keys = sorted(required_v55_keys - set(v55))
require(
    not missing_v55_keys,
    f'Notebook 55 verdict lacks required fields: {missing_v55_keys}',
)

N53 = (
    Path(NOTEBOOK53_RUN_OVERRIDE)
    if NOTEBOOK53_RUN_OVERRIDE
    else Path(v55['notebook53_run'])
)
N09 = Path(v55['notebook09_run'])

require(N53.exists(), 'Notebook 53 source recorded by Notebook 55 is unavailable.')
require(N09.exists(), 'Notebook 09 source recorded by Notebook 55 is unavailable.')
require((N09 / 'proposals').exists(), f'Notebook 09 proposals directory missing: {N09 / "proposals"}')

compare = pd.read_csv(N55 / 'tables' / 'shadow_page_comparison.csv')
require('page' in compare.columns, 'Notebook 55 comparison lacks page column.')
compare['page'] = compare['page'].astype(str)
require(not compare['page'].duplicated().any(), 'Notebook 55 comparison has duplicate page rows.')

pages = sorted(compare['page'].tolist())
require(len(pages) > 0, 'Pilot roster is empty.')

policy_id = v55['policy_id']
implementation_id = v55['implementation_id']

print('Notebook 55:', N55)
print('Notebook 53:', N53)
print('Notebook 09:', N09)
print('Pages:', len(pages))
print('Policy:', policy_id)
print('Implementation:', implementation_id)


## 2. Load the exact page pairs and initialise durable review state

The Notebook 09 side is loaded from `proposals/`, matching the completed Notebook 55 algorithmic shadow comparison. The `final/` directory is intentionally not used.


In [ ]:
def incumbent_path(page):
    hits = [
        path
        for path in (N09 / 'proposals').glob(page + '.*')
        if path.is_file() and path.suffix.lower() in SUPPORTED
    ]
    require(
        len(hits) == 1,
        f'{page}: expected one Notebook 09 proposal, found {hits}',
    )
    return hits[0]


def v88_path(page):
    path = N53 / 'pages' / page / 'corrected.png'
    require(path.exists(), f'{page}: V8.8 corrected page missing: {path}')
    return path


expected = compare.set_index('page')
STATE = {}
identity_rows = []

for page in pages:
    n09_path = incumbent_path(page)
    candidate_path = v88_path(page)

    notebook09 = imread(n09_path)
    v88 = imread(candidate_path)
    observed = imread(N53 / 'pages' / page / 'observed.png')
    m3 = imread(N53 / 'pages' / page / 'm3.png') > 0
    safe_remove = imread(N53 / 'pages' / page / 'safe_remove.png') > 0

    shapes = {
        'notebook09': notebook09.shape,
        'v88': v88.shape,
        'observed': observed.shape,
        'm3': m3.shape,
        'safe_remove': safe_remove.shape,
    }
    require(
        len(set(shapes.values())) == 1,
        f'{page}: shape mismatch: {shapes}',
    )

    disagreement = int(np.sum(notebook09 != v88))

    if 'different_pixels' in expected.columns:
        expected_disagreement = int(expected.loc[page, 'different_pixels'])
        require(
            disagreement == expected_disagreement,
            (
                f'{page}: Notebook 56 disagreement {disagreement} '
                f'does not match Notebook 55 {expected_disagreement}'
            ),
        )
    else:
        expected_disagreement = disagreement

    STATE[page] = {
        'status': 'unreviewed',
        'notes': '',
        'history': [],
        'notebook09': notebook09,
        'v88': v88,
        'observed': observed,
        'm3': m3,
        'safe_remove': safe_remove,
        'notebook09_path': str(n09_path),
        'v88_path': str(candidate_path),
        'notebook09_file_sha256': sha256_file(n09_path),
        'v88_file_sha256': sha256_file(candidate_path),
        'notebook09_array_sha256': array_hash(notebook09),
        'v88_array_sha256': array_hash(v88),
        'disagreement_pixels': disagreement,
        'expected_notebook55_disagreement_pixels': expected_disagreement,
    }

    identity_rows.append({
        'page': page,
        'notebook09_path': str(n09_path),
        'v88_path': str(candidate_path),
        'notebook09_file_sha256': sha256_file(n09_path),
        'v88_file_sha256': sha256_file(candidate_path),
        'notebook09_array_sha256': array_hash(notebook09),
        'v88_array_sha256': array_hash(v88),
        'disagreement_pixels': disagreement,
        'notebook55_disagreement_pixels': expected_disagreement,
        'exact_notebook55_replay': disagreement == expected_disagreement,
    })

pd.DataFrame(identity_rows).to_csv(
    A / 'source_identity_and_notebook55_replay.csv',
    index=False,
)


def persist(page, event):
    state = STATE[page]
    atomic_json(
        DEC / f'{page}_decision.json',
        {
            'page': page,
            'status': state['status'],
            'notes': state['notes'],
            'history': state['history'],
            'event': event,
            'policy_id': policy_id,
            'implementation_id': implementation_id,
            'notebook55_run': str(N55),
            'notebook53_run': str(N53),
            'notebook09_run': str(N09),
            'notebook09_path': state['notebook09_path'],
            'v88_path': state['v88_path'],
            'notebook09_file_sha256': state['notebook09_file_sha256'],
            'v88_file_sha256': state['v88_file_sha256'],
            'notebook09_array_sha256': state['notebook09_array_sha256'],
            'v88_array_sha256': state['v88_array_sha256'],
            'disagreement_pixels': state['disagreement_pixels'],
            'updated_utc': datetime.now(timezone.utc).isoformat(),
        },
    )


for page in pages:
    persist(page, {'event': 'initialise'})

print('Loaded and replay-checked pages:', len(STATE))


## 3. Reviewer functions

Overlay rules:

- **Red:** Notebook 09 and V8.8 differ.
- **Green:** SAFE_REMOVE where the outputs agree.
- Red takes priority where disagreement overlaps SAFE_REMOVE.


In [ ]:
def counts():
    values = [state['status'] for state in STATE.values()]
    return {
        key: values.count(key)
        for key in [
            'unreviewed',
            'accept_v88',
            'retain_notebook09',
            'defer',
            'mask_review_required',
        ]
    }


def pair_view(page, overlay=True):
    state = STATE[page]
    left = cv2.cvtColor(state['notebook09'], cv2.COLOR_GRAY2RGB)
    right = cv2.cvtColor(state['v88'], cv2.COLOR_GRAY2RGB)

    if overlay:
        disagreement = state['notebook09'] != state['v88']
        safe_remove_only = state['safe_remove'] & ~disagreement
        right[safe_remove_only] = (40, 190, 70)
        right[disagreement] = (255, 40, 40)

    divider = np.full((left.shape[0], 8, 3), 235, np.uint8)
    combined = np.hstack([left, divider, right])

    cv2.putText(
        combined,
        'NOTEBOOK 09 PROPOSAL',
        (12, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        .62,
        (30, 30, 30),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        combined,
        'V8.8 REGISTERED REPLAY',
        (left.shape[1] + 20, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        .62,
        (30, 30, 30),
        2,
        cv2.LINE_AA,
    )

    return combined


def info(page):
    state = STATE[page]
    queue = counts()
    return (
        f'{page} | status={state["status"]} | '
        f'disagreement={state["disagreement_pixels"]} pixels | '
        f'SAFE_REMOVE={int(state["safe_remove"].sum())} | '
        f'queue={queue}'
    )


def refresh(page, overlay=True):
    return pair_view(page, overlay), info(page), STATE[page]['notes']


def decide(page, status, notes, overlay=True):
    require(
        status in [
            'accept_v88',
            'retain_notebook09',
            'defer',
            'mask_review_required',
        ],
        'Invalid decision',
    )

    state = STATE[page]
    state['status'] = status
    state['notes'] = notes
    state['history'].append({
        'utc': datetime.now(timezone.utc).isoformat(),
        'status': status,
        'notes': notes,
    })
    persist(page, {'event': 'decision', 'status': status})
    return refresh(page, overlay)


def approve_all(notes):
    queue = counts()
    require(
        queue['unreviewed'] == 0 and queue['defer'] == 0,
        f'Pilot cannot close: {queue}',
    )

    rows = []

    for page, state in STATE.items():
        if state['status'] == 'accept_v88':
            output = state['v88']
            source = 'V8.8_registered_replay'
        elif state['status'] in [
            'retain_notebook09',
            'mask_review_required',
        ]:
            output = state['notebook09']
            source = 'Notebook09_proposal_fallback'
        else:
            raise RuntimeError(f'{page}: unresolved status {state["status"]}')

        path = FINAL / f'{page}.png'
        require(cv2.imwrite(str(path), output), f'{page}: final write failed')

        rows.append({
            'page': page,
            'decision': state['status'],
            'approved_source': source,
            'notes': state['notes'],
            'final_path': str(path),
            'final_sha256': sha256_file(path),
            'v88_disagreement_pixels': state['disagreement_pixels'],
            'safe_remove_pixels': int(state['safe_remove'].sum()),
            'notebook09_source_path': state['notebook09_path'],
            'v88_source_path': state['v88_path'],
        })

    result = pd.DataFrame(rows)
    result.to_csv(T / 'pilot_page_decisions.csv', index=False)

    atomic_json(
        OUT / 'pilot_approval.json',
        {
            'approved_utc': datetime.now(timezone.utc).isoformat(),
            'approval_notes': notes,
            'policy_id': policy_id,
            'implementation_id': implementation_id,
            'notebook55_run': str(N55),
            'notebook53_run': str(N53),
            'notebook09_run': str(N09),
            'notebook09_comparison_source': 'proposals',
            'counts': counts(),
            'pages': len(result),
            'v88_accepted': int(
                (result.approved_source == 'V8.8_registered_replay').sum()
            ),
            'notebook09_retained': int(
                (result.approved_source == 'Notebook09_proposal_fallback').sum()
            ),
            'automatic_approval': False,
        },
    )

    return f'Pilot closed. {counts()}'


## 4. Supervised reviewer


In [ ]:
first = pages[0]

with gr.Blocks(title='V8.8E Supervised Registered-Replay Pilot') as app:
    gr.Markdown(
        '# V8.8E supervised registered-replay pilot\n'
        'Left: Notebook 09 proposal. Right: V8.8 registered replay. '
        'Red indicates output disagreement. Green indicates SAFE_REMOVE '
        'where the outputs agree. Red takes priority where they overlap. '
        'Human approval remains mandatory.'
    )

    with gr.Row():
        page = gr.Dropdown(pages, value=first, label='Page')
        overlay = gr.Checkbox(
            True,
            label='Show disagreement / SAFE_REMOVE overlay',
        )

    view = gr.Image(height=760, label='Notebook 09 proposal | V8.8')
    status = gr.Textbox(interactive=False, label='Page and queue status')
    notes = gr.Textbox(label='Decision notes')

    with gr.Row():
        accept = gr.Button('Accept V8.8 correction', variant='primary')
        retain = gr.Button('Retain Notebook 09 proposal')
        defer = gr.Button('Defer')
        maskreview = gr.Button('Require mask review')

    gr.Markdown('## Close pilot')
    approval_notes = gr.Textbox(label='Pilot approval notes')
    close = gr.Button('Write approved pages and close pilot', variant='primary')
    close_status = gr.Textbox(interactive=False)

    page.change(refresh, [page, overlay], [view, status, notes])
    overlay.change(refresh, [page, overlay], [view, status, notes])
    app.load(refresh, [page, overlay], [view, status, notes])

    accept.click(
        lambda p, n, o: decide(p, 'accept_v88', n, o),
        [page, notes, overlay],
        [view, status, notes],
    )
    retain.click(
        lambda p, n, o: decide(p, 'retain_notebook09', n, o),
        [page, notes, overlay],
        [view, status, notes],
    )
    defer.click(
        lambda p, n, o: decide(p, 'defer', n, o),
        [page, notes, overlay],
        [view, status, notes],
    )
    maskreview.click(
        lambda p, n, o: decide(p, 'mask_review_required', n, o),
        [page, notes, overlay],
        [view, status, notes],
    )
    close.click(approve_all, [approval_notes], [close_status])

app.launch(share=PUBLIC_LINK, debug=False, show_error=True)


## 5. Run after the reviewer closes the pilot


In [ ]:
require(
    (OUT / 'pilot_approval.json').exists(),
    'Pilot is not closed. Complete decisions and use the close button first.',
)

decisions = pd.read_csv(T / 'pilot_page_decisions.csv')
approval = read_json(OUT / 'pilot_approval.json')

require(
    not decisions['page'].astype(str).duplicated().any(),
    'Pilot decision table contains duplicate pages.',
)

final_rows = []
for page in pages:
    path = FINAL / f'{page}.png'
    require(path.exists(), f'Missing final page: {path}')
    final_rows.append({
        'page': page,
        'file': path.name,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })

final_inventory = pd.DataFrame(final_rows)
final_inventory.to_csv(T / 'final_page_inventory.csv', index=False)

gates = {
    'notebook55_pass': v55.get('verdict') == EXPECTED_N55_VERDICT,
    'notebook55_first_failed_gate_clear': v55.get('first_failed_gate') is None,
    'all_pages_decided': len(decisions) == len(pages),
    'decision_roster_exact': set(decisions.page.astype(str)) == set(pages),
    'no_unreviewed': counts()['unreviewed'] == 0,
    'no_deferred': counts()['defer'] == 0,
    'final_pages_complete': len(final_inventory) == len(pages),
    'human_approval_recorded': True,
    'notebook55_disagreement_replay_exact': bool(
        pd.read_csv(A / 'source_identity_and_notebook55_replay.csv')
        ['exact_notebook55_replay'].all()
    ),
}

first_failed = next((key for key, value in gates.items() if value is not True), None)
verdict = (
    'PASS_SUPERVISED_PILOT_COMPLETE'
    if first_failed is None
    else 'FAIL_V8_8E_PILOT'
)

record = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'verdict': verdict,
    'first_failed_gate': first_failed,
    'gates': gates,
    'policy_id': policy_id,
    'implementation_id': implementation_id,
    'notebook55_run': str(N55),
    'notebook53_run': str(N53),
    'notebook09_run': str(N09),
    'notebook09_comparison_source': 'proposals',
    'pages': len(pages),
    'v88_accepted': int(
        (decisions.approved_source == 'V8.8_registered_replay').sum()
    ),
    'notebook09_retained': int(
        (decisions.approved_source == 'Notebook09_proposal_fallback').sum()
    ),
    'mask_review_required': int(
        (decisions.decision == 'mask_review_required').sum()
    ),
    'automatic_approval': False,
    'automatic_output_authority': False,
    'promotion_authority': False,
    'meaning': (
        'Completion records supervised pilot outcomes. A separate release '
        'decision is required before changing the primary correction engine.'
    ),
}

atomic_json(OUT / 'notebook56_verdict.json', record)
print(json.dumps(record, indent=2))

if first_failed:
    raise RuntimeError(f'Notebook 56 failed at {first_failed}')


## 6. Curated upload package


In [ ]:
summary = [
    'NOTEBOOK 56 / V8.8E SUPERVISED REGISTERED-REPLAY PILOT',
    '',
    f'Verdict: {verdict}',
    f'Pages: {len(pages)}',
    f'V8.8 accepted: {record["v88_accepted"]}',
    f'Notebook 09 proposals retained: {record["notebook09_retained"]}',
    f'Mask review required: {record["mask_review_required"]}',
    f'Policy ID: {policy_id}',
    f'Implementation ID: {implementation_id}',
    f'Notebook 55 run: {N55}',
    f'Notebook 53 run: {N53}',
    f'Notebook 09 run: {N09}',
    '',
    'Human final approval was mandatory.',
    'Notebook 09 comparison source was proposals/.',
    'This notebook has no production-promotion authority.',
]

(OUT / 'executive_summary.txt').write_text(chr(10).join(summary))

files = [
    OUT / 'executive_summary.txt',
    OUT / 'notebook56_verdict.json',
    OUT / 'pilot_approval.json',
    T / 'pilot_page_decisions.csv',
    T / 'final_page_inventory.csv',
    A / 'source_identity_and_notebook55_replay.csv',
] + sorted(DEC.glob('*.json'))

rows = []
for src in files:
    require(src.exists(), f'Missing upload artefact {src}')
    relative_group = (
        'decisions' if src.parent == DEC
        else 'tables' if src.parent == T
        else 'audit' if src.parent == A
        else 'root'
    )
    dst_name = src.name if relative_group == 'root' else f'{relative_group}__{src.name}'
    dst = U / dst_name
    shutil.copy2(src, dst)
    rows.append({
        'file': dst.name,
        'source': str(src),
        'bytes': dst.stat().st_size,
        'sha256': sha256_file(dst),
    })

pd.DataFrame(rows).to_csv(U / 'upload_manifest.csv', index=False)

(U / 'README.txt').write_text(
    'Notebook 56 V8.8E supervised registered-replay pilot decision pack.\n'
    'Notebook 09 source: proposals/.\n'
    'Full-resolution approved pages remain in the run final/ directory and '
    'are indexed by tables/final_page_inventory.csv.\n'
)

(U / 'upload_file_list.txt').write_text(
    chr(10).join(sorted(path.name for path in U.iterdir() if path.is_file()))
    + chr(10)
)

zip_path = U / 'notebook56_v8_8e_upload_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in U.iterdir():
        if path.is_file() and path != zip_path:
            archive.write(path, path.name)

require(zip_path.stat().st_size > 0, 'Upload ZIP failed')
print('Upload:', zip_path)
